# 11 — A/B Test: Does a Retention Intervention Actually Work?

The churn model (notebook 04) identifies WHO is likely to leave. This notebook
answers the question a business actually needs answered next: if you DO something
about it — a retention discount, proactive outreach, a feature unlock — does it
actually reduce churn, and how would you know for sure rather than guess from a
before/after comparison that could just as easily be explained by seasonality, a
concurrent marketing push, or regression to the mean among a self-selected risky
group?

This notebook demonstrates the experiment framework in `src/experiment.py`:
- Correctly sizing an experiment *before* running it, so a null result can't be
  wrongly blamed on "no effect" when it was actually an underpowered test
- Randomly assigning the churn model's highest-risk customers to treatment/control
- A proper two-proportion z-test with a confidence interval on the effect size, not
  just a p-value
- A plain-language summary suitable for a non-technical stakeholder

Since there's no real intervention data available, the experiment's outcome is
**simulated with a known true effect** — the same honesty discipline used to validate
the drift detector (Section 10 deliberately introduced a known shift to prove PSI
catches it). This lets the statistical analysis be checked against ground truth: does
the test correctly recover an effect we know we put in?


In [1]:
import sys
sys.path.insert(0, '../')

import pandas as pd
from src.preprocessing import preprocess, get_model_features
from src.model import train_xgboost
from src.experiment import (
    simulate_retention_experiment, analyze_experiment,
    required_sample_size, summarize_result
)

df = preprocess()
X, y = get_model_features(df)
model = train_xgboost(X, y)
df['churn_probability'] = model.predict_proba(X)[:, 1]
print('Model trained and scored full customer base.')


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [14:53:35] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Model trained and scored full customer base.


## Step 1: size the experiment BEFORE running it

If the intervention's true effect is genuinely as small as, say, a 5-point reduction
in churn probability, how many customers per arm would we need to reliably detect
that, rather than just guessing at a sample size or using "whatever customers we
have"?


In [2]:
baseline_rate = df['churn_probability'].quantile(0.70)  # rough proxy for the high-risk population's baseline
print(f'Approximate baseline churn rate for the targeted high-risk segment: {baseline_rate:.1%}')
print()

for mde in [0.03, 0.05, 0.08, 0.10]:
    n = required_sample_size(baseline_rate=baseline_rate, minimum_detectable_effect=mde)
    print(f'To detect a {mde:.0%}-point reduction: {n:,} customers needed per arm ({n*2:,} total)')


Approximate baseline churn rate for the targeted high-risk segment: 71.0%

To detect a 3%-point reduction: 3,700 customers needed per arm (7,400 total)
To detect a 5%-point reduction: 1,355 customers needed per arm (2,710 total)
To detect a 8%-point reduction: 542 customers needed per arm (1,084 total)
To detect a 10%-point reduction: 352 customers needed per arm (704 total)


## Step 2: define the experiment population

Target the model's highest-risk customers — the top 30% by predicted churn
probability — since a retention intervention is both most valuable and most likely to
show a measurable effect on customers who are actually at meaningful risk, rather
than diluting the experiment with low-risk customers who were never going to churn
regardless of any intervention.


In [3]:
high_risk = df[df['churn_probability'] >= df['churn_probability'].quantile(0.70)].copy()
print(f'Experiment population: {len(high_risk):,} high-risk customers')
print(f'Average predicted churn probability in this group: {high_risk["churn_probability"].mean():.1%}')


Experiment population: 2,113 high-risk customers
Average predicted churn probability in this group: 79.6%


## Step 3: simulate the experiment

The intervention's TRUE effect is set explicitly here (an 8-percentage-point absolute
reduction in churn probability) — this is the ground truth the statistical test below
needs to recover, the same validation logic used for the drift detector.


In [4]:
TRUE_EFFECT = 0.08  # ground truth, known only because this is a simulation

experiment_df = simulate_retention_experiment(
    high_risk, true_treatment_effect=TRUE_EFFECT, seed=42
)
experiment_df[['customerID', 'churn_probability', 'arm', 'churned_in_experiment']].head(10)


,customerID,churn_probability,arm,churned_in_experiment
0,CUST-00002,0.728464,treatment,True
1,CUST-00003,0.765585,control,True
2,CUST-00008,0.790548,treatment,False
3,CUST-00009,0.770016,treatment,True
4,CUST-00012,0.912384,control,True
5,CUST-00013,0.867458,treatment,True
6,CUST-00015,0.794896,treatment,True
7,CUST-00016,0.759382,treatment,True
8,CUST-00018,0.774496,control,True
9,CUST-00019,0.804622,control,True


## Step 4: analyze the results properly

A two-proportion z-test, with a 95% confidence interval on the effect size — not just
a p-value. The p-value alone answers "is this effect distinguishable from zero";
the confidence interval answers the more useful business question, "how big is the
effect, and how confident are we in that range."


In [5]:
result = analyze_experiment(experiment_df)
print(summarize_result(result))
print()
print(f'True effect used in simulation: {TRUE_EFFECT:.1%}')
print(f'Effect recovered by the statistical test: {result.absolute_effect:.1%}')
print(f'Does the 95% CI contain the true effect? {result.ci_low <= TRUE_EFFECT <= result.ci_high}')


The intervention showed a statistically significant effect on churn (p = 0.0000, alpha = 0.05). Control arm churn rate: 81.7% (n=1072). Treatment arm churn rate: 71.1% (n=1041). The intervention reduced churn by 10.6% percentage points (+13.0% relative change), 95% CI: [7.0%, 14.2%].

True effect used in simulation: 8.0%
Effect recovered by the statistical test: 10.6%
Does the 95% CI contain the true effect? True


## Step 5: what if the experiment had been underpowered?

To make the sample-size step's importance concrete rather than abstract, re-run the
same analysis on a deliberately small random subsample — this simulates what would
have happened if the experiment had been run without first checking Step 1's
required sample size.


In [6]:
small_sample = experiment_df.sample(n=min(60, len(experiment_df)), random_state=7)
small_result = analyze_experiment(small_sample)
print(summarize_result(small_result))
print()
print(
    'Even with the same true 8-point effect actually present in the data, '
    'a too-small sample can fail to reach statistical significance -- '
    'which is exactly why sizing the experiment correctly BEFORE running it '
    '(Step 1) matters more than analyzing it carefully after the fact.'
)


The intervention showed not a statistically significant effect on churn (p = 0.3711, alpha = 0.05). Control arm churn rate: 70.0% (n=30). Treatment arm churn rate: 80.0% (n=30). The intervention increased churn by 10.0% percentage points (-14.3% relative change), 95% CI: [-31.8%, 11.8%].

Even with the same true 8-point effect actually present in the data, a too-small sample can fail to reach statistical significance -- which is exactly why sizing the experiment correctly BEFORE running it (Step 1) matters more than analyzing it carefully after the fact.


## Takeaway

The properly-sized experiment correctly detects and quantifies the simulated
retention intervention's effect, with a confidence interval that contains the true
value used to generate the data — a direct validation that the statistical framework
works as intended, not just that it runs without errors. The underpowered-sample
comparison makes a genuinely important, often-skipped point concrete: a business
that runs a retention experiment on too few customers and finds "no significant
effect" may not have learned that the intervention doesn't work — they may have
learned nothing at all, because the test was never capable of detecting an effect of
that size to begin with. Sizing the experiment correctly before running it is not a
formality; it's what makes a null result actually mean something.
